# 2. Building a plant

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pbtamarona/openpytea_workshop/blob/main/notebooks/02_building_a_plant.ipynb)

A `Plant` combines an equipment list with economic assumptions and returns everything a TEA report needs:
fixed capital investment, fixed and variable operating costs, revenue, yearly cash flows, and the profitability
metrics (NPV, IRR, ROI, payback time, levelized cost). In this notebook we build the green-ammonia plant step by
step and learn how to read and change its assumptions.

## Learning goals

- Configure a `Plant` with financial assumptions, products, and variable OPEX inputs.
- Compute the CAPEX, OPEX, revenue, and yearly cash flow.
- Interpret the profitability metrics, and the assumptions behind them.
- Change assumptions with `update_configuration()` and recalculate.

## 0) Setup: rebuild the equipment list from notebook 01

In [ ]:
# Run this cell. It installs OpenPyTEA if it is missing (e.g., on Google Colab).
try:
    import openpytea
except ImportError:
    import subprocess, sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "openpytea"])
    import openpytea

print("OpenPyTEA version:", openpytea.__version__)

In [ ]:
from openpytea import Equipment

# Equipment of the green-ammonia loop (from notebook 01)
compressor  = Equipment(name="C-101 Syngas compressor", param=1_200,
                        process_type="Fluids",
                        category="Compressors, fans, & blowers",
                        type="Compressor, centrifugal", material="Carbon steel")
motor       = Equipment(name="M-101 Compressor motor", param=1_200 / 0.90,
                        process_type="Electrical",
                        category="Motors & generators",
                        type="Totally enclosed motor")
exchanger   = Equipment(name="E-101 Feed/effluent exchanger", param=250,
                        process_type="Fluids",
                        category="Heat exchangers", type="U-tube shell & tube",
                        material="304 stainless steel")
converter   = Equipment(name="R-101 Ammonia converter", param=12_000,
                        process_type="Fluids",
                        category="Reactors", type="Tubular fixed bed",
                        material="304 stainless steel")
separator   = Equipment(name="V-101 Ammonia separator", param=25,
                        process_type="Fluids",
                        category="Pressure vessels", type="Vertical",
                        material="Carbon steel")
chiller     = Equipment(name="K-101 Refrigeration unit", param=800,
                        process_type="Fluids",
                        category="Utilities",
                        type="Packaged mechanical refrigerator")
product_pump = Equipment(name="P-101 Product pump", param=20,
                         process_type="Fluids",
                         category="Pumps", type="Single-stage centrifugal",
                         material="Carbon steel")

equipment_list = [compressor, motor, exchanger, converter, separator, chiller,
                  product_pump]

print(f"Total direct cost: "
      f"{sum(e.direct_cost for e in equipment_list):,.0f} USD")

## 1) The plant configuration

A `Plant` is created from one dictionary. Only `plant_name`, `process_type`, `equipment`, and `plant_products`
are strictly required; everything else has a documented default. Units are up to you, but they must be
consistent: OpenPyTEA multiplies **daily** quantities by 365 and by the plant utilization to get yearly values,
and prices are per unit of that quantity.

In [ ]:
from openpytea import Plant

ammonia_config = {
    "plant_name": "Green Ammonia",
    # "Solids" | "Fluids" | "Mixed": sets the default OSBL, D&E, and
    # contingency factors
    "process_type": "Fluids",
    # location factor lookup (see Plant.locFactors); some countries also
    # need a "region"
    "country": "Netherlands",
    "equipment": equipment_list,

    # Financial assumptions
    "interest_rate": 0.08,               # discount rate
    "project_lifetime": 20,              # years, including construction
    "plant_utilization": 0.90,           # fraction of the year on stream
    "tax_rate": 0.25,

    # Products: the FIRST entry is the main product used for the levelized cost
    "plant_products": {
        "ammonia": {"production": 100_000, "price": 0.95},   # kg/day, USD/kg
    },

    # Variable OPEX: daily consumption and unit price of every feed and utility
    "variable_opex_inputs": {
        # kg/day,  USD/kg
        "hydrogen":    {"consumption": 17_800, "price": 2.00},
        # kg/day,  USD/kg
        "nitrogen":    {"consumption": 82_400, "price": 0.05},
        # kWh/day, USD/kWh
        "electricity": {"consumption": 60_000, "price": 0.08},
    },
}

ammonia_plant = Plant(ammonia_config)

## 2) Fixed capital investment

$$\text{FCI} = \text{ISBL} \cdot (1 + OS) \cdot (1 + D\&E + X) \cdot LF$$

ISBL is the sum of the equipment direct costs. The offsite (OS), design and engineering (D&E), and contingency (X) factors default to 0.3, 0.3, and 0.1 for fluid plants (Towler & Sinnott). The location factor (LF) comes from the country (and region). Any of these can be overridden.

In [ ]:
ammonia_plant.calculate_fixed_capital(print_results=True)

In [ ]:
# The location factors OpenPyTEA knows about
Plant.locFactors

## 3) Operating costs and revenue

**Variable OPEX**: consumption × price × 365 × utilization, summed over `variable_opex_inputs`. **Fixed OPEX**:
operating labor (operators estimated from the process steps, or set by you), supervision, overheads, maintenance
(5 % of ISBL), taxes and insurance, and more (Turton et al.). **Revenue**: production × price × 365 × utilization,
summed over all products.

In [ ]:
ammonia_plant.calculate_variable_opex(print_results=True)

In [ ]:
ammonia_plant.calculate_fixed_opex(print_results=True)
print(f"Operators hired (auto-estimated): "
      f"{ammonia_plant.calculate_operators_hired()}")

In [ ]:
ammonia_plant.calculate_revenue(print_results=True)

## 4) Cash flow and profitability metrics

`calculate_cash_flow()` builds the year-by-year table. By default:

- **Capital spending:** spread over three construction years (30 / 60 / 10 %).
- **Production ramp-up:** 0 / 0 / 40 / 80 / 100 %.
- **Depreciation:** straight-line over 15 years, starting in the first production year.
- **Taxes:** paid with a one-year lag.
- **Working capital:** 15 % of FCI by default, drawn at start-up and released at the end.

In [ ]:
ammonia_plant.calculate_cash_flow(print_results=True)

Once the cash flows exist, the metrics follow:

| Metric | Meaning | Method |
|---------|-------------------------------------------------------|------------------------------------|
| NPV | Present value of all cash flows at the interest rate | `calculate_npv()` |
| LCOP | Levelized cost of product: the selling price at which NPV = 0 (before tax), after crediting by-products | `calculate_levelized_cost()` |
| PBT | Payback time: the year in which the cumulative (undiscounted) cash flow turns positive | `calculate_payback_time()` |
| ROI | Average yearly net profit over total investment | `calculate_roi()` |
| IRR | Discount rate at which NPV = 0 | `calculate_irr()` |

`calculate_all()` runs all the calculation steps of sections 2 to 4 (fixed capital, variable and fixed OPEX,
revenue, cash flow, and then the metrics) in the same order. You can also call a single step on its own: it
reruns the steps it depends on first, so its result always matches the current configuration.

In [ ]:
ammonia_plant.calculate_all()

print(f"Fixed capital investment : {ammonia_plant.fixed_capital:>14,.0f} USD")
print(f"Variable OPEX            : "
      f"{ammonia_plant.variable_production_costs:>14,.0f} USD/yr")
print(f"Fixed OPEX               : "
      f"{ammonia_plant.fixed_production_costs:>14,.0f} USD/yr")
print(f"Revenue                  : {ammonia_plant.revenue:>14,.0f} USD/yr")
print(f"Levelized cost of ammonia: "
      f"{ammonia_plant.levelized_cost:>14.3f} USD/kg")
print(f"Net present value        : {ammonia_plant.npv:>14,.0f} USD")
print(f"Internal rate of return  : {ammonia_plant.irr:>14.1%}")
print(f"Return on investment     : {ammonia_plant.roi:>14.1f} %")
print(f"Payback time             : {ammonia_plant.payback_time:>14.1f} years")

> **Reading the result.** The levelized cost (0.824 USD/kg) is below the assumed selling price of 0.95 USD/kg,
> so the project is profitable: the NPV is about 19 MUSD, the IRR (13.7 %) is well above the 8 % discount rate,
> and the plant pays back after about 9 years. Hydrogen is by far the largest cost item. Keep these numbers in
> mind: the next notebooks ask how robust this conclusion is.

## 5) Changing assumptions

`update_configuration()` merges a partial dictionary into the configuration (nested dictionaries included), so you
can change one number and recalculate. Almost every default can be overridden: the location factor (`loc_factor`),
the number of operators (`operators_hired`), fixed OPEX items such as `maintenance` or `supervision`, and the cash-flow assumptions `capex_ramp`, `production_ramp`, and
`working_capital`.

In [ ]:
from copy import deepcopy

variant = deepcopy(ammonia_plant)
variant.update_configuration({
    "plant_name": "Green Ammonia (cheaper H2)",
    # price only; consumption is kept
    "variable_opex_inputs": {"hydrogen": {"price": 1.50}},
    "fixed_opex_factors": {"maintenance": 0.03},  # 3 % of ISBL instead of 5 %
})
variant.calculate_all()

print(f"LCOP: {ammonia_plant.levelized_cost:.3f} -> "
      f"{variant.levelized_cost:.3f} USD/kg")
print(f"NPV : {ammonia_plant.npv:,.0f} -> {variant.npv:,.0f} USD")

## 6) Task 2.1: where should the plant be built?

Location factors scale the ISBL to local construction costs. Compare the fixed capital investment and the levelized cost of the ammonia plant for these locations: (1) **Netherlands**; (2) **United States**; region `"Gulf Coast"`, (3) **China**, region `"indigenous"` (locally sourced equipment); and (4) **India**

> **Tip:** loop over `deepcopy(ammonia_plant)` and `update_configuration({"country": ..., "region": ...})`;
> use `"region": None` for countries without regions.

In [ ]:
# YOUR CODE HERE

> **Note:** this is an oversimplification. Only the location factor for construction costs changes here, but in reality the levelized cost also depends on local feedstock and utility prices, labor rates, tax rates, the cost of capital, and the plant utilization (e.g., the local capacity factor of renewable power).

## 7) Task 2.2: the effect of a by-product

The plant recovers the reaction heat as steam and sells 80,000 kg/day (0.8 kg per kg of ammonia) at 0.025 USD/kg. Add the steam as a second product and compare the revenue, levelized cost and NPV.

> **Tip:** the first entry in `plant_products` is the main product. Every other entry is a by-product: its revenue is added to the cash flow and credited against the cost of the main product.

In [ ]:
# YOUR CODE HERE

## 8) Task 2.3 (bonus): the break-even price

By definition the levelized cost is the price at which NPV = 0 **before taxes**. Check it: set the ammonia price
equal to the computed `levelized_cost` on a copy of the plant and print the NPV. It comes out negative, because
the 25 % tax is still paid on the profitable years. Now also set `"tax_rate": 0` and confirm that the NPV
becomes zero.

In [ ]:
# YOUR CODE HERE

## 9) Task 2.4 (bonus): the effect of catalyst replacement

The converter catalyst must be replaced every 5 years (in years 7, 12 and 17) at 1.2 MUSD each time. Add these
replacement costs to the plant, recalculate, and report how much the NPV and levelized cost change. Also show
the cash flow table and check that the replacement costs appear in the right years. To find out how, see the
[OpenPyTEA documentation](https://openpytea.readthedocs.io) or
[part 2 of the OpenPyTEA walkthrough](https://github.com/pbtamarona/OpenPyTEA/blob/main/walkthrough/part_2_plant.ipynb).

In [ ]:
# YOUR CODE HERE

## Recap

- One configuration dictionary defines a `Plant`; the first product sets the levelized cost.
- `calculate_all()` gives CAPEX, OPEX, cash flow, and all metrics in one call.
- `deepcopy` + `update_configuration()` creates scenarios.

**Next:** [3. Cost breakdowns and sensitivity](03_cost_breakdowns_and_sensitivity.ipynb)